# eCommerce EDA — Normalized (Parquet)

Runs on the **normalized** star schema produced by `04_normalize.py`:

| table | rows | grain |
|-------|------|-------|
| `categories` | 691 | category_id → category_code |
| `products` | 206,876 | product_id → category_id, brand |
| `events` (fact) | ~110M | one row per event (view/cart/purchase) |

Parquet is columnar + compressed (14GB CSV → 3.4GB), so scans are faster than the raw CSV and brand/category come from small dimension tables via join.

**Heavy aggregation → DuckDB; charts → pandas/matplotlib.**

## Setup — load the star schema

In [1]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Resolve data/normalized relative to repo root (works from root or eda/).
ROOT = Path.cwd()
for cand in (ROOT, ROOT.parent):
    if (cand / "data" / "normalized").is_dir():
        ROOT = cand
        break
NORM = ROOT / "data" / "normalized"

need = ["events.parquet", "products.parquet", "categories.parquet"]
missing = [f for f in need if not (NORM / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing {missing} in {NORM}. Run `python eda/04_normalize.py` first."
    )

con = duckdb.connect()  # in-memory
# Load as TABLES (not views) so repeated queries hit RAM, not disk.
for name in ("events", "products", "categories"):
    con.sql(
        f"CREATE TABLE {name} AS "
        f"SELECT * FROM read_parquet('{(NORM / (name + '.parquet')).as_posix()}')"
    )

def q(sql: str) -> pd.DataFrame:
    return con.sql(sql).df()

print("Loaded:")
display(q("""
    SELECT 'events' AS tbl, count(*) AS rows FROM events
    UNION ALL SELECT 'products', count(*) FROM products
    UNION ALL SELECT 'categories', count(*) FROM categories
"""))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded:


,tbl,rows
0,events,109950743
1,products,206876
2,categories,691


## 1. Overview

In [2]:
q("""
    SELECT
        count(*)                     AS total_events,
        min(event_time)              AS start_time,
        max(event_time)              AS end_time,
        count(DISTINCT user_id)      AS unique_users,
        count(DISTINCT product_id)   AS unique_products,
        count(DISTINCT user_session) AS unique_sessions
    FROM events
""").T

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,0
total_events,109950743
start_time,2019-10-01 00:00:00
end_time,2019-11-30 23:59:59
unique_users,5316649
unique_products,206876
unique_sessions,23016650
